In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# 0. Path configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected project structure:
    Project_root/
    ├─ Code/
    │  └─ classifier/
    ├─ Data/
    │  └─ ASIGranite.xls
    └─ Result/
    """
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists():
            return path

    if start_dir.name.lower() == "classifier":
        return start_dir.parents[1]

    if start_dir.name.lower() == "code":
        return start_dir.parent

    return start_dir


PROJECT_ROOT = find_project_root()

RESULT_BASE_DIR = PROJECT_ROOT / "Result"

DATA_ROOT = (
    RESULT_BASE_DIR
    / "01_Foldwise_Preprocessing_and_Ratio_Features"
)

STABILITY_SUMMARY_FILE = (
    RESULT_BASE_DIR
    / "05_Stable_Cluster_Champions_and_Novel_Ratio_Candidates"
    / "rho090_stable_champions_and_ratio_candidates_summary.xlsx"
)

OUT_DIR = (
    RESULT_BASE_DIR
    / "08_Fixed_FiveFold_Multimodel_Performance_Comparison"
    / "LogisticRegression"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

IMPUTATION_METHODS = ["global_mean", "knn"]
N_OUTER_FOLDS = 5
CLASS_ORDER = ["A", "S", "I"]

TOP_N_STABLE_RATIOS = 20
SEED = 42

MODEL_NAME = "LogisticRegression"


# ============================================================
# 1. Utility functions
# ============================================================

def normalize_type_value(value):
    """
    Standardize A-type, S-type, and I-type labels to A, S, and I.
    """
    s = str(value).strip()

    if s in ["A", "A-type", "A-Type", "A_TYPE", "A type"]:
        return "A"

    if s in ["S", "S-type", "S-Type", "S_TYPE", "S type"]:
        return "S"

    if s in ["I", "I-type", "I-Type", "I_TYPE", "I type"]:
        return "I"

    return s


def display_feature_name(name):
    """
    Format feature names for display only.
    """
    s = str(name)
    s = s.replace("R_Major_", "")
    s = s.replace("R_Trace_", "")
    s = s.replace("A12O3", "Al2O3")
    s = s.replace("10000*Ga/Al", "10000xGa/Al")
    s = s.replace("10000*Ga/A1", "10000xGa/Al")
    s = s.replace("10000×Ga/Al", "10000xGa/Al")

    return s


def feature_key(name):
    """
    Build a normalized key for matching feature names across files.
    """
    s = str(name).strip()
    s = display_feature_name(s)
    s = s.replace("×", "*")
    s = re.sub(r"\s+", "", s)

    return s.lower()


def resolve_one_feature(feature, columns):
    """
    Resolve one feature name against available dataframe columns.
    """
    columns = [str(c).strip() for c in columns]

    if feature in columns:
        return feature

    target_key = feature_key(feature)

    for col in columns:
        if feature_key(col) == target_key:
            return col

    fixed = str(feature).replace("Al2O3", "A12O3").replace("A12O3", "Al2O3")

    if fixed in columns:
        return fixed

    fixed_key = feature_key(fixed)

    for col in columns:
        if feature_key(col) == fixed_key:
            return col

    return None


def resolve_feature_list(features, columns):
    """
    Resolve a list of feature names against available dataframe columns.
    """
    resolved = []
    missing = []

    for feature in features:
        col = resolve_one_feature(feature, columns)

        if col is None:
            missing.append(feature)
        else:
            if col not in resolved:
                resolved.append(col)

    return resolved, missing


def get_classical_features(columns):
    """
    Get classical or commonly used granite-discrimination features
    if they are available in the current fold.
    """
    aliases = {
        "A/CNK": ["A/CNK", "ACNK"],
        "A/NK": ["A/NK", "ANK"],
        "10000xGa/Al": ["10000*Ga/Al", "10000xGa/Al", "10000×Ga/Al", "10000*Ga/A1"],
        "Zr+Nb+Ce+Y": ["Zr+Nb+Ce+Y", "Zr_Nb_Ce_Y"],
        "Sr/Y": ["Sr/Y", "R_Trace_Sr/Y"],
        "Rb/Sr": ["Rb/Sr", "R_Trace_Rb/Sr"],
        "K2O/Na2O": ["K2O/Na2O", "R_Major_K2O/Na2O"],
        "Fe2O3t/MgO": ["Fe2O3t/MgO", "R_Major_Fe2O3t/MgO"],
    }

    selected = []

    for _, options in aliases.items():
        for feature in options:
            col = resolve_one_feature(feature, columns)

            if col is not None:
                selected.append(col)
                break

    return list(dict.fromkeys(selected))


def load_stable_novel_ratios():
    """
    Load stable candidate novel ratios from the Script 05 summary.
    """
    if not STABILITY_SUMMARY_FILE.exists():
        raise FileNotFoundError(
            f"Stability summary file was not found: {STABILITY_SUMMARY_FILE}"
        )

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    if "stable_ratio_candidates" not in xls.sheet_names:
        raise ValueError(
            "The stability summary file does not contain the "
            "'stable_ratio_candidates' sheet."
        )

    df = pd.read_excel(
        STABILITY_SUMMARY_FILE,
        sheet_name="stable_ratio_candidates"
    )

    if "Feature" not in df.columns:
        raise ValueError(
            "The 'stable_ratio_candidates' sheet does not contain a Feature column."
        )

    if "Appearance_count" in df.columns:
        df = df[df["Appearance_count"] >= 6].copy()

    sort_cols = []
    ascending = []

    for col in [
        "Appearance_count",
        "Mean_champion_score",
        "Mean_SHAP_importance",
        "Mean_TopK_frequency"
    ]:
        if col in df.columns:
            sort_cols.append(col)
            ascending.append(False)

    if sort_cols:
        df = df.sort_values(sort_cols, ascending=ascending)

    return df["Feature"].astype(str).head(TOP_N_STABLE_RATIOS).tolist()


def load_stable_feature_set():
    """
    Load the stable feature set from the Script 05 summary.

    Multiple sheet names are supported for compatibility with previous
    output versions.
    """
    if not STABILITY_SUMMARY_FILE.exists():
        return []

    xls = pd.ExcelFile(STABILITY_SUMMARY_FILE)

    candidate_sheets = [
        "stable_feature_candidates",
        "stable_features_for_paper",
        "core_stable_features",
        "champion_stability",
        "champion_stability_summary"
    ]

    for sheet_name in candidate_sheets:
        if sheet_name in xls.sheet_names:
            df = pd.read_excel(STABILITY_SUMMARY_FILE, sheet_name=sheet_name)

            if "Feature" in df.columns:
                return df["Feature"].dropna().astype(str).tolist()

    return []


def read_fold_data(method, fold):
    """
    Read one fixed outer-fold train/test dataset.
    """
    method_dir = DATA_ROOT / method

    train_path = method_dir / f"fold_{fold:02d}_train_with_ratios.xlsx"
    test_path = method_dir / f"fold_{fold:02d}_test_with_ratios.xlsx"

    if not train_path.exists():
        raise FileNotFoundError(f"Training-fold file was not found: {train_path}")

    if not test_path.exists():
        raise FileNotFoundError(f"Test-fold file was not found: {test_path}")

    train_df = pd.read_excel(train_path)
    test_df = pd.read_excel(test_path)

    train_df.columns = [str(c).strip() for c in train_df.columns]
    test_df.columns = [str(c).strip() for c in test_df.columns]

    if TYPE_COL not in train_df.columns:
        raise ValueError(f"Label column '{TYPE_COL}' was not found in: {train_path}")

    if TYPE_COL not in test_df.columns:
        raise ValueError(f"Label column '{TYPE_COL}' was not found in: {test_path}")

    train_df[TYPE_COL] = train_df[TYPE_COL].apply(normalize_type_value)
    test_df[TYPE_COL] = test_df[TYPE_COL].apply(normalize_type_value)

    return train_df, test_df


def build_feature_sets(train_columns):
    """
    Build the feature sets evaluated by this classifier.
    """
    stable_ratios = load_stable_novel_ratios()
    stable_features = load_stable_feature_set()

    classical = get_classical_features(train_columns)
    novel, novel_missing = resolve_feature_list(stable_ratios, train_columns)
    stable_set, stable_missing = resolve_feature_list(stable_features, train_columns)

    feature_sets = {
        "Classical_only": classical,
        "Stable_novel_ratios": novel,
        "Classical_plus_stable_novel": list(dict.fromkeys(classical + novel)),
        "Stable_feature_set": stable_set,
    }

    feature_sets = {
        name: features
        for name, features in feature_sets.items()
        if len(features) > 0
    }

    return feature_sets


def make_model():
    """
    Build the Logistic Regression model with the specified parameters.
    """
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            multi_class="multinomial",
            solver="lbfgs",
            C=1.0,
            max_iter=5000,
            n_jobs=-1,
            class_weight="balanced",
            random_state=SEED
        ))
    ])


def evaluate_one(y_true, y_pred):
    """
    Calculate overall multiclass metrics.
    """
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "Accuracy": acc,
        "Balanced_accuracy": bacc,
        "Macro_precision": macro_precision,
        "Macro_recall": macro_recall,
        "Macro_F1": macro_f1,
        "Weighted_precision": weighted_precision,
        "Weighted_recall": weighted_recall,
        "Weighted_F1": weighted_f1,
    }


def plot_summary(summary_df):
    """
    Plot mean Macro-F1 values for all feature sets.
    """
    if summary_df.empty:
        return

    metric = "Macro_F1_mean"

    if metric not in summary_df.columns:
        return

    plot_df = summary_df.sort_values(metric, ascending=True)

    labels = (
        plot_df["Method"].astype(str)
        + " | "
        + plot_df["Feature_set"].astype(str)
    )

    plt.figure(figsize=(10, max(5, 0.35 * len(plot_df))), dpi=300)
    plt.barh(labels, plot_df[metric])
    plt.xlabel("Mean Macro-F1", fontsize=12, fontweight="bold")
    plt.ylabel("Method | Feature set", fontsize=12, fontweight="bold")
    plt.title(
        f"{MODEL_NAME}: Fixed Five-Fold Performance",
        fontsize=14,
        fontweight="bold"
    )
    plt.tight_layout()

    out_png = OUT_DIR / f"{MODEL_NAME}_macroF1_summary.png"
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


# ============================================================
# 2. Main workflow
# ============================================================

def main():
    all_metric_rows = []
    all_classwise_rows = []
    all_cm_rows = []

    for method in IMPUTATION_METHODS:
        print(f"\n========== {MODEL_NAME} | {method} ==========")

        for fold in range(1, N_OUTER_FOLDS + 1):
            print(f"Outer fold {fold}")

            train_df, test_df = read_fold_data(method, fold)
            feature_sets = build_feature_sets(train_df.columns)

            for feature_set_name, features in feature_sets.items():
                print(f"  Feature set: {feature_set_name}, n_features={len(features)}")

                X_train = train_df[features].replace([np.inf, -np.inf], np.nan)
                y_train = train_df[TYPE_COL].astype(str)

                X_test = test_df[features].replace([np.inf, -np.inf], np.nan)
                y_test = test_df[TYPE_COL].astype(str)

                model = make_model()
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)

                row = {
                    "Model": MODEL_NAME,
                    "Method": method,
                    "Outer_fold": fold,
                    "Feature_set": feature_set_name,
                    "N_features": len(features),
                    "Features": "; ".join(features),
                    "N_train": len(train_df),
                    "N_test": len(test_df),
                }

                row.update(evaluate_one(y_test, y_pred))
                all_metric_rows.append(row)

                report = classification_report(
                    y_test,
                    y_pred,
                    labels=CLASS_ORDER,
                    output_dict=True,
                    zero_division=0
                )

                for cls in CLASS_ORDER:
                    all_classwise_rows.append({
                        "Model": MODEL_NAME,
                        "Method": method,
                        "Outer_fold": fold,
                        "Feature_set": feature_set_name,
                        "Class": cls,
                        "Precision": report[cls]["precision"],
                        "Recall": report[cls]["recall"],
                        "F1": report[cls]["f1-score"],
                        "Support": report[cls]["support"],
                    })

                cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)

                for i, true_cls in enumerate(CLASS_ORDER):
                    for j, pred_cls in enumerate(CLASS_ORDER):
                        all_cm_rows.append({
                            "Model": MODEL_NAME,
                            "Method": method,
                            "Outer_fold": fold,
                            "Feature_set": feature_set_name,
                            "True_class": true_cls,
                            "Pred_class": pred_cls,
                            "Count": int(cm[i, j]),
                        })

    metrics_df = pd.DataFrame(all_metric_rows)
    classwise_df = pd.DataFrame(all_classwise_rows)
    cm_df = pd.DataFrame(all_cm_rows)

    metric_cols = [
        "Accuracy",
        "Balanced_accuracy",
        "Macro_precision",
        "Macro_recall",
        "Macro_F1",
        "Weighted_precision",
        "Weighted_recall",
        "Weighted_F1",
    ]

    summary_df = (
        metrics_df
        .groupby(["Model", "Method", "Feature_set", "N_features"], as_index=False)[metric_cols]
        .agg(["mean", "std"])
    )

    summary_df.columns = [
        "_".join([c for c in col if c])
        for col in summary_df.columns.to_flat_index()
    ]

    summary_df = summary_df.reset_index()

    out_xlsx = OUT_DIR / f"{MODEL_NAME}_fixed5fold_results.xlsx"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        metrics_df.to_excel(
            writer,
            sheet_name="fold_metrics",
            index=False
        )

        summary_df.to_excel(
            writer,
            sheet_name="summary_mean_std",
            index=False
        )

        classwise_df.to_excel(
            writer,
            sheet_name="classwise_metrics",
            index=False
        )

        cm_df.to_excel(
            writer,
            sheet_name="confusion_matrix_long",
            index=False
        )

    plot_summary(summary_df)

    print("\n========== Current model summary ==========")
    show_cols = [
        "Model",
        "Method",
        "Feature_set",
        "N_features",
        "Accuracy_mean",
        "Balanced_accuracy_mean",
        "Macro_precision_mean",
        "Macro_recall_mean",
        "Macro_F1_mean",
        "Weighted_F1_mean"
    ]
    show_cols = [c for c in show_cols if c in summary_df.columns]

    summary_to_show = (
        summary_df[show_cols]
        .sort_values(["Method", "Macro_F1_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )
    print(summary_to_show.to_string(index=False))

    print("\n========== Best result for the current model ==========")
    best_row = summary_df.sort_values("Macro_F1_mean", ascending=False).head(1)
    print(best_row[show_cols].to_string(index=False))

    summary_png = OUT_DIR / f"{MODEL_NAME}_macroF1_summary.png"

    print("\nCompleted:", MODEL_NAME)
    print("Result file:", out_xlsx)
    print("Figure file:", summary_png)
    print("Output directory:", OUT_DIR)


# ============================================================
# 3. Program entry point
# ============================================================

if __name__ == "__main__":
    main()